# 03 – Model Training & Evaluation

This notebook trains four classifiers on the CKD dataset, compares their performance, and saves the best model bundle.

In [2]:
import sys, os
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model  import LogisticRegression
from sklearn.tree          import DecisionTreeClassifier
from sklearn.ensemble      import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics       import (accuracy_score, precision_score, recall_score,
                                   f1_score, confusion_matrix, classification_report)

from utils.preprocessing   import preprocess_pipeline

TRAIN_PATH = '../dataset/Training_CKD_dataset.csv'
TEST_PATH  = '../dataset/Testing_CKD_dataset.csv'
MODEL_PATH = '../model/kidney_model.pkl'

print('Libraries loaded OK')

ModuleNotFoundError: No module named 'utils.preprocessing'

## 1. Preprocessing Pipeline

In [ ]:
X_train, X_test, y_train, y_test, encoders, scaler, class_names = \
    preprocess_pipeline(TRAIN_PATH, TEST_PATH)

print(f'Training samples  : {X_train.shape[0]:,}')
print(f'Testing  samples  : {X_test.shape[0]:,}')
print(f'Number of features: {X_train.shape[1]}')
print(f'Classes           : {class_names}')

## 2. Define & Train Classifiers

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42, n_jobs=-1),
    'Decision Tree':       DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}

for name, clf in models.items():
    print(f'Training {name} ...', end=' ', flush=True)
    t0 = time.time()
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    elapsed = round(time.time() - t0, 2)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    results[name] = {
        'accuracy':  acc,
        'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall':    recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1_score':  f1,
        'model':     clf,
        'y_pred':    y_pred,
        'time_s':    elapsed,
    }
    print(f'done in {elapsed}s  |  Acc={acc:.4f}  F1={f1:.4f}')

## 3. Model Comparison

In [ ]:
metric_df = pd.DataFrame({
    name: {
        'Accuracy':  r['accuracy'],
        'Precision': r['precision'],
        'Recall':    r['recall'],
        'F1-Score':  r['f1_score'],
    }
    for name, r in results.items()
}).T

print('Model Comparison:')
display(metric_df.style.format('{:.4f}').background_gradient(cmap='YlGn', axis=0))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
metric_df.plot(kind='bar', ax=ax, colormap='tab10', edgecolor='white', width=0.7)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontweight='bold', fontsize=13)
ax.legend(loc='lower right')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
plt.tight_layout()
os.makedirs('../assets', exist_ok=True)
plt.savefig('../assets/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Best Model Selection

In [ ]:
best_name  = max(results, key=lambda n: results[n]['f1_score'])
best_res   = results[best_name]
best_model = best_res['model']
y_pred     = best_res['y_pred']

print(f'Best Model : {best_name}')
print(f'Accuracy   : {best_res["accuracy"]:.4f}')
print(f'Precision  : {best_res["precision"]:.4f}')
print(f'Recall     : {best_res["recall"]:.4f}')
print(f'F1-Score   : {best_res["f1_score"]:.4f}')

## 5. Classification Report

In [ ]:
report = classification_report(y_test, y_pred, target_names=class_names, zero_division=0)
print(f'Classification Report -- {best_name}:')
print(report)

## 6. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual',    fontsize=11)
ax.set_title(f'Confusion Matrix -- {best_name}', fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right', fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,  fontsize=8)
plt.tight_layout()
plt.savefig('../assets/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Feature Importance (Tree-based Models)

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=X_train.columns)
    importances = importances.sort_values(ascending=True).tail(20)
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.barh(importances.index, importances.values,
            color=plt.cm.viridis(np.linspace(0.2, 0.85, len(importances))),
            edgecolor='none')
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'Top 20 Feature Importances -- {best_name}', fontweight='bold')
    plt.tight_layout()
    plt.savefig('../assets/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'{best_name} does not expose feature_importances_.')

## 8. Save the Best Model Bundle

In [ ]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)

comparison_metrics = {
    name: {k: v for k, v in res.items() if k not in ('model', 'y_pred')}
    for name, res in results.items()
}

bundle = {
    'model':                 best_model,
    'model_name':            best_name,
    'encoders':              encoders,
    'scaler':                scaler,
    'classes':               class_names,
    'features':              list(X_train.columns),
    'feature_names':         list(X_train.columns),
    'metrics': {
        'accuracy':          best_res['accuracy'],
        'precision':         best_res['precision'],
        'recall':            best_res['recall'],
        'f1_score':          best_res['f1_score'],
    },
    'comparison_metrics':    comparison_metrics,
    'confusion_matrix':      cm,
    'classification_report': report,
    'class_distribution':    train_raw['Target'].value_counts().to_dict(),
    'train_shape':           train_raw.shape,
    'test_shape':            test_raw.shape,
}

os.makedirs('../model', exist_ok=True)
joblib.dump(bundle, MODEL_PATH)
print(f'Model bundle saved to: {MODEL_PATH}')

## 9. Verify Saved Model

In [ ]:
loaded = joblib.load(MODEL_PATH)
print('Bundle loaded successfully!')
print(f'  Model name : {loaded["model_name"]}')
print(f'  Accuracy   : {loaded["metrics"]["accuracy"]:.4f}')
print(f'  F1-Score   : {loaded["metrics"]["f1_score"]:.4f}')
print(f'  Classes    : {loaded["classes"]}')
print(f'  Features   : {len(loaded["features"])} columns')